# Learning QAOA — from a cost function to a quantum solve

**`qcoptlib` learning series · notebook 1 of a few (QAOA → Grover → QML)**

A thin, runnable walk-through of the **Quantum Approximate Optimization Algorithm**: what it is,
how a classical cost function becomes a quantum circuit, and how `qcoptlib` runs one end-to-end on
a local simulator. Distilled from the QUBIT × AT&T hackathon QAOA workshop (Classiq portfolio
optimization) and re-grounded in this library's Qiskit backend so every cell runs without an
account.

> **Status: in build.** Part of the notes I'm keeping as I build the quantum-optimization library;
> the companion Grover and QML notebooks follow.


## 1. What QAOA is, in three ingredients

QAOA is a **hybrid** algorithm: a shallow parametrised quantum circuit proposes solutions, a
classical optimiser tunes its angles. For a cost function $C(x)$ over bitstrings $x$:

1. **Start in uniform superposition** — a Hadamard on every qubit puts equal amplitude on all
   $2^n$ candidate solutions: $|+\rangle^{\otimes n}$.
2. **Alternate two layers, $p$ times** (depth $p$):
   - **Cost layer** $e^{-i\gamma C}$ — rotates each basis state's *phase* by its cost, so good and
     bad solutions become distinguishable by phase. Encoded from $C(x)$ (details in §3).
   - **Mixer layer** $e^{-i\beta B}$ with $B=\sum_i X_i$ — $R_X(\beta)$ on every qubit, which
     *moves amplitude between* solutions so interference can concentrate it on the good ones.
3. **Optimise the angles** $(\gamma_1,\beta_1,\dots,\gamma_p,\beta_p)$ with a classical optimiser
   to minimise the measured expected cost, then sample the tuned circuit and read out the best
   bitstring.

As $p\to\infty$ QAOA approaches the exact optimum (it can emulate adiabatic evolution); at small
$p$ on a simulator it's a heuristic — good, not guaranteed. That honesty matters (see §6).


## 2. The cost function is an Ising Hamiltonian

The cost layer needs $C(x)$ as a quantum operator. Any QUBO
$C(x)=\sum_i h_i x_i + \sum_{i<j} J_{ij} x_i x_j + \text{const}$ over $x_i\in\{0,1\}$ becomes an
**Ising** cost over spins $s_i\in\{-1,+1\}$ via $x_i=(1-s_i)/2$, giving single-$Z$ and $Z Z$ terms.
That $Z/ZZ$ operator *is* the cost Hamiltonian $C$ whose phase the cost layer applies. `qcoptlib`
does this conversion for you:


In [ ]:
import numpy as np
from qcoptlib.qubo import QUBO

# a tiny 3-variable QUBO: C(x) = 2 x0  - 3 x0 x1 + x1 x2 + 1
q = QUBO.zeros(3)
q.add_linear(0, 2.0).add_quadratic(0, 1, -3.0).add_quadratic(1, 2, 1.0).add_const(1.0)

h_ising, J_ising, offset = q.to_ising()   # x = (1 - s)/2  ->  Z / ZZ coefficients
print("single-Z (h):", np.round(h_ising, 3))
print("ZZ (J):\n", np.round(J_ising, 3))
print("offset:", round(offset, 3))
print("brute-force optimum:", q.brute_force())   # ground truth on small n


## 3. The cost layer, two ways

There are two common ways to turn $C(x)$ into the cost-layer circuit — worth knowing both, because
the hackathon used the first and this library uses the second.

**(a) Arithmetic `phase` (Classiq).** You write the cost as ordinary arithmetic on quantum
variables and the platform builds $e^{i\gamma C}$ for you — no manual Pauli algebra. This is the
hackathon workshop's approach (portfolio optimization):

```python
# Classiq (needs an account to run) — the cost layer is one call:
@qfunc
def main(params: CArray[CReal, 2*NUM_LAYERS], w: Output[PortfolioVars]) -> None:
    allocate(w); hadamard_transform(w)
    repeat(NUM_LAYERS, lambda i: (
        phase(objective(w, returns, cov, lam), params[2*i]),   # cost layer  e^{i γ C}
        mixer_layer(params[2*i + 1], w),                        # mixer       RX(β)
    ))
```

**(b) Pauli operator (Qiskit).** You hand QAOA the cost Hamiltonian as a `SparsePauliOp` of
$Z/ZZ$ terms (from §2) and use the built-in `qaoa_ansatz`. This is what
`qcoptlib.quantum.qiskit_backend` does — fully local, no account:

```python
from qcoptlib.quantum.qiskit_backend import qubo_to_sparse_pauli
cost_op = qubo_to_sparse_pauli(q)     # the Z/ZZ Hamiltonian
# qaoa_ansatz(cost_operator=cost_op, reps=p) builds Hadamards + p × (cost, mixer) layers
```

Same algorithm, two front-ends: `phase` is friendlier for arithmetic objectives with integer
variables; the Pauli form is transparent and runs anywhere. `qcoptlib` ships the Qiskit path and a
matching Classiq backend behind one interface.


## 4. Run one end-to-end (local, Qiskit) — Max-Cut

The textbook QAOA problem: partition a graph's nodes into two sets cutting as many edges as
possible. For an edge $(i,j)$, the term $-\tfrac12(1 - s_i s_j)$ is $-1$ when the endpoints are on
opposite sides — so *maximising* cut edges is *minimising* $\sum_{(i,j)} \tfrac12(s_i s_j - 1)$,
a QUBO. We build it directly (a learning example; real problems get a builder in `src/`), then let
`qcoptlib` solve it and check against brute force.


In [ ]:
from qcoptlib.qubo import QUBO

def maxcut_qubo(n, edges):
    """Minimising this QUBO maximises the number of cut edges (x_i = which side)."""
    q = QUBO.zeros(n)
    # cut(i,j) = x_i + x_j - 2 x_i x_j  (1 iff endpoints differ); maximise sum -> minimise -sum
    for i, j in edges:
        q.add_linear(i, -1.0).add_linear(j, -1.0).add_quadratic(i, j, 2.0)
    return q

# a 5-node ring with one chord (its max cut is computed below, not assumed)
edges = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)]
q = maxcut_qubo(5, edges)
best_bits, best_energy = q.brute_force()
print(f"brute-force cut assignment {best_bits}, edges cut = {int(-best_energy)}")


In [ ]:
from qcoptlib.quantum.cache import cached_qaoa

# solve the SAME QUBO with QAOA (Aer). cached_qaoa stores the result so re-runs are instant.
res = cached_qaoa(q, num_layers=3, maxiter=150, shots=4096, restarts=2, seed=0,
                  cache_dir="qaoa_cache")
cut_qaoa = int(-q.energy(res.best_bits))
print(f"QAOA assignment {tuple(res.best_bits)}, edges cut = {cut_qaoa}  "
      f"({'optimal' if cut_qaoa == int(-best_energy) else 'suboptimal'}), "
      f"{len(res.history)} optimiser evals")


In [ ]:
import matplotlib.pyplot as plt
from qcoptlib.viz import plot_convergence

fig, ax = plt.subplots(figsize=(7, 4))
plot_convergence(res.history, ax=ax, title="QAOA convergence (Max-Cut)")
plt.tight_layout(); plt.show()


## 5. Constraints — penalties and slack

QAOA is *unconstrained*, so a hard constraint is added as a **quadratic penalty** that is zero on
feasible states and positive otherwise — then the objective and penalty simply add (QUBOs compose).

- **Equality** $\sum_i a_i x_i = t$ → penalty $P\,(\sum_i a_i x_i - t)^2$.
- **Inequality** $\sum_i a_i x_i \le t$ → introduce a non-negative **slack** $s$ and penalise
  $P\,(\sum_i a_i x_i + s - t)^2$ (the slack absorbs the gap; it costs extra qubits).

`qcoptlib.qubo` provides `equality_penalty`, `onehot_penalty`, and `inequality_padding`. Here: pick
the two highest-value of four items — *exactly two* (an equality constraint). The unconstrained
objective would grab all four; the penalty forces feasibility.


In [ ]:
from qcoptlib.qubo import QUBO, equality_penalty

values = [5, 4, 3, 1]
objective = QUBO.zeros(4)
for i, v in enumerate(values):
    objective.add_linear(i, -float(v))          # reward selecting item i (negate to minimise)

print("unconstrained optimum:", objective.brute_force()[0], "(picks everything)")

# constraint: choose exactly 2 items  ->  penalty P·(Σ x_i − 2)²,  P large enough to dominate
constrained = objective + equality_penalty(4, [1, 1, 1, 1], target=2, weight=100.0)
bits = constrained.brute_force()[0]
print("with 'pick exactly 2' penalty:", bits, "-> items", [i for i, b in enumerate(bits) if b])


In [ ]:
# and the SAME constrained QUBO through QAOA (feasibility now comes from the penalty)
res_c = cached_qaoa(constrained, num_layers=3, maxiter=150, shots=4096, seed=0,
                    cache_dir="qaoa_cache")
picked = [i for i, b in enumerate(res_c.best_bits) if b]
print(f"QAOA picked items {picked} (feasible: {len(picked) == 2})")


## 6. Knobs and honest intuition

- **Depth $p$ (`num_layers`)** — more layers = more expressive, deeper circuit, harder to optimise.
- **Penalty weight $P$** — too small and infeasible states win; too large and it flattens the
  objective so QAOA can't see it. Set it just above the largest objective swing a violation buys.
- **Angle initialisation** — an adiabatic-inspired schedule ($\gamma:0\to1$, $\beta:1\to0$) beats
  random starts; `qcoptlib.quantum.common.adiabatic_init` provides it, and `restarts>1` adds random
  restarts on top.
- **Optimiser** — COBYLA (derivative-free) is the default; its per-evaluation trace is *non-monotone*
  (it explores uphill), so `qcoptlib` reads out the **best angles seen**, not the last iterate.
- **Read-out** — score every sampled bitstring by the real cost and keep the best
  (`best_bits_from_counts`), so a weakly-concentrated distribution still yields a good answer.
- **Always brute-force small instances** as ground truth; report the QAOA gap honestly.

### Try it
Change `num_layers`, the penalty `weight`, the graph/edges, or the optimiser, and watch the
convergence curve and the cut/feasibility change.


## References
- Farhi, Goldstone, Gutmann, *A Quantum Approximate Optimization Algorithm*, arXiv:1411.4028 (2014).
- Lucas, *Ising formulations of many NP problems*, Front. Phys. (2014).
- Glover, Kochenberger, Du, *A Tutorial on Formulating and Using QUBO Models* (2019).
- Classiq docs — QAOA and the `phase` function (the hackathon workshop's platform).

*Library used here: `qcoptlib.qubo` (build/compose/convert), `qcoptlib.quantum` (QAOA + cache),
`qcoptlib.viz` (convergence). All solves are cached to `qaoa_cache/`.*
